# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/octane21/flyrank-internship-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
dataset = pd.read_csv('dataset.csv')


dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45019 entries, 0 to 45018
Data columns (total 77 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Unnamed: 0                       45019 non-null  int64  
 1   client_hash_id                   45019 non-null  object 
 2   content_hash_id                  45019 non-null  object 
 3   month                            45019 non-null  object 
 4   gsc_impressions_30d              45019 non-null  int64  
 5   gsc_clicks_30d                   45019 non-null  int64  
 6   gsc_sum_position_30d             45019 non-null  int64  
 7   ga4_pageviews_30d                45019 non-null  int64  
 8   ga4_sessions_30d                 45019 non-null  int64  
 9   ga4_users_30d                    45019 non-null  int64  
 10  ga4_engaged_sessions_30d         45019 non-null  int64  
 11  sessions_ai_30d                  45019 non-null  int64  
 12  sessions_organic_3

In [14]:
features = [
    'client_hash_id',
    'gsc_impressions_30d',
    'gsc_clicks_30d',
    'gsc_sum_position_30d',
    'ga4_pageviews_30d',
    'ga4_sessions_30d',
    'ga4_users_30d',
    'ga4_engaged_sessions_30d',
    'sessions_ai_30d',
    'sessions_organic_30d',
    'scroll_events_30d',
    'ga4_data_available',
    'gsc_data_available',
    'gsc_avg_position_30d',
    'gsc_impressions_prev_30d',
    'gsc_impressions_2m_ago_30d',
    'gsc_clicks_prev_30d',
    'gsc_clicks_2m_ago_30d',
    'gsc_sum_position_prev_30d',
    'gsc_sum_position_2m_ago_30d',
    'ga4_pageviews_prev_30d',
    'ga4_pageviews_2m_ago_30d',
    'ga4_sessions_prev_30d',
    'ga4_sessions_2m_ago_30d',
    'ga4_users_prev_30d',
    'ga4_users_2m_ago_30d',
    'ga4_engaged_sessions_prev_30d',
    'ga4_engaged_sessions_2m_ago_30d',
    'sessions_ai_prev_30d',
    'sessions_ai_2m_ago_30d',
    'sessions_organic_prev_30d',
    'sessions_organic_2m_ago_30d',
    'scroll_events_prev_30d',
    'scroll_events_2m_ago_30d',
    'gsc_avg_position_prev_30d',
    'gsc_avg_position_2m_ago_30d',
    'content_type',
    'search_volume',
    'competition',
    'competition_level',
    'cpc',
    'main_intent',
    'backlinks',
    'category_count',
    'word_count',
    'char_count',
    'content_age_days',
    'ctr_30d',
    'engagement_rate_30d',
    'scroll_rate_30d',
    'ai_traffic_30d',
    'decline_future_label',
    'position_tier_30d',
     'age_tier_30d',
     'age_tier_order_30d',
     'word_count_tier_30d',
    'char_count_tier_30d',
 'impression_tier_30d'
]


dataset_90d = dataset[features].copy()

dataset_90d.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45019 entries, 0 to 45018
Data columns (total 58 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   client_hash_id                   45019 non-null  object 
 1   gsc_impressions_30d              45019 non-null  int64  
 2   gsc_clicks_30d                   45019 non-null  int64  
 3   gsc_sum_position_30d             45019 non-null  int64  
 4   ga4_pageviews_30d                45019 non-null  int64  
 5   ga4_sessions_30d                 45019 non-null  int64  
 6   ga4_users_30d                    45019 non-null  int64  
 7   ga4_engaged_sessions_30d         45019 non-null  int64  
 8   sessions_ai_30d                  45019 non-null  int64  
 9   sessions_organic_30d             45019 non-null  int64  
 10  scroll_events_30d                45019 non-null  int64  
 11  ga4_data_available               45019 non-null  bool   
 12  gsc_data_available

In [15]:
numeric_data = dataset_90d.select_dtypes(
    include=['number']
)

dataset_90d[numeric_data.columns] = (
    numeric_data.replace(
        [np.inf, -np.inf],
        np.nan
    )
)

In [16]:
categorical_cols = [
    'content_type',
    'competition_level',
    'main_intent',
    'position_tier_30d',
     'age_tier_30d',
     'word_count_tier_30d',
    'char_count_tier_30d',
 'impression_tier_30d'
]

for col in categorical_cols:
    dataset_90d[col] = dataset_90d[col].astype('category')

In [17]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold


target_col = 'decline_future_label'
group_col = 'client_hash_id'

X = dataset_90d.drop(
    columns=[target_col, group_col]
)

y = dataset_90d[target_col]

groups = dataset_90d[group_col]

assert groups.notna().all(), "client_hash_id contains missing values"

assert set(y.unique()).issubset({0, 1}), "Target must be binary"

outer_gkf = GroupKFold(n_splits=5)

inner_gkf = GroupKFold(n_splits=4)

split_results = []

for outer_fold, (remaining_idx, test_idx) in enumerate(
    outer_gkf.split(X, y, groups=groups),
    start=1
):

    # -----------------------------------------------------
    # Outer split
    # -----------------------------------------------------

    X_remaining = X.iloc[remaining_idx]
    y_remaining = y.iloc[remaining_idx]
    groups_remaining = groups.iloc[remaining_idx]

    X_test = X.iloc[test_idx]
    y_test = y.iloc[test_idx]
    groups_test = groups.iloc[test_idx]

    remaining_clients = groups_remaining.unique()

    inner_splits = list(
        inner_gkf.split(
            remaining_clients,
            groups=remaining_clients
        )
    )

    inner_fold = (outer_fold - 1) % 4

    train_client_idx, val_client_idx = inner_splits[inner_fold]

    train_clients = remaining_clients[train_client_idx]
    val_clients = remaining_clients[val_client_idx]

    train_mask = groups_remaining.isin(train_clients)
    val_mask = groups_remaining.isin(val_clients)

    train_idx = remaining_idx[train_mask.to_numpy()]
    val_idx = remaining_idx[val_mask.to_numpy()]

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    groups_train = groups.iloc[train_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]
    groups_val = groups.iloc[val_idx]

    X_test = X.iloc[test_idx]
    y_test = y.iloc[test_idx]
    groups_test = groups.iloc[test_idx]

    assert set(groups_train).isdisjoint(set(groups_val))
    assert set(groups_train).isdisjoint(set(groups_test))
    assert set(groups_val).isdisjoint(set(groups_test))

    split_results.append({
        'fold': outer_fold,

        'train_idx': train_idx,
        'val_idx': val_idx,
        'test_idx': test_idx,

        'train_clients': groups_train.unique(),
        'val_clients': groups_val.unique(),
        'test_clients': groups_test.unique()
    })

    print(f"\n========== OUTER FOLD {outer_fold} ==========")

    print(f"Train rows     : {len(train_idx):,}")
    print(f"Validation rows: {len(val_idx):,}")
    print(f"Test rows     : {len(test_idx):,}")

    print(f"Train clients : {groups_train.nunique():,}")
    print(f"Val clients  : {groups_val.nunique():,}")
    print(f"Test clients : {groups_test.nunique():,}")

    print(
        f"Train ratio  : "
        f"{groups_train.nunique() / groups.nunique():.1%}"
    )

    print(
        f"Val ratio    : "
        f"{groups_val.nunique() / groups.nunique():.1%}"
    )

    print(
        f"Test ratio   : "
        f"{groups_test.nunique() / groups.nunique():.1%}"
    )

    print("\nTarget distribution:")

    print("Train:")
    print(y_train.value_counts(normalize=True))

    print("\nValidation:")
    print(y_val.value_counts(normalize=True))

    print("\nTest:")
    print(y_test.value_counts(normalize=True))


fold_1 = split_results[0]

train_idx = fold_1['train_idx']
val_idx = fold_1['val_idx']
test_idx = fold_1['test_idx']

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]

X_val = X.iloc[val_idx]
y_val = y.iloc[val_idx]

X_test = X.iloc[test_idx]
y_test = y.iloc[test_idx]



========== OUTER FOLD 1 ==========
Train rows     : 16,902
Validation rows: 1,261
Test rows     : 26,856
Train clients : 13
Val clients  : 5
Test clients : 1
Train ratio  : 68.4%
Val ratio    : 26.3%
Test ratio   : 5.3%

Target distribution:
Train:
decline_future_label
0    0.607443
1    0.392557
Name: proportion, dtype: float64

Validation:
decline_future_label
0    0.638382
1    0.361618
Name: proportion, dtype: float64

Test:
decline_future_label
0    0.565907
1    0.434093
Name: proportion, dtype: float64

========== OUTER FOLD 2 ==========
Train rows     : 33,441
Validation rows: 199
Test rows     : 11,379
Train clients : 13
Val clients  : 5
Test clients : 1
Train ratio  : 68.4%
Val ratio    : 26.3%
Test ratio   : 5.3%

Target distribution:
Train:
decline_future_label
0    0.546724
1    0.453276
Name: proportion, dtype: float64

Validation:
decline_future_label
0    0.552764
1    0.447236
Name: proportion, dtype: float64

Test:
decline_future_label
0    0.69224
1    0.30776
Name:

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


from xgboost import XGBClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

param_grid = {
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [200, 400],
    'min_child_weight': [1, 5],
    'gamma': [0],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'reg_alpha': [0],
    'reg_lambda': [1]
}

parameter_combinations = list(
    ParameterGrid(param_grid)
)

print(
    "Jumlah kombinasi hyperparameter:",
    len(parameter_combinations)
)

def build_xgb_model(params, y_train):

    negative_count = (y_train == 0).sum()
    positive_count = (y_train == 1).sum()

    if positive_count == 0:
        raise ValueError(
            "Data training tidak memiliki kelas positif."
        )

    scale_pos_weight = (
        negative_count / positive_count
    )

    model = XGBClassifier(
        **params,

        objective='binary:logistic',

        scale_pos_weight=scale_pos_weight,

        eval_metric='aucpr',

        tree_method='hist',
        enable_categorical=True,

        random_state=42,

        n_jobs=-1,

        verbosity=0
    )

    return model

def evaluate_model(model, X_eval, y_eval):

    y_proba = model.predict_proba(X_eval)[:, 1]

    y_pred = (y_proba >= 0.5).astype(int)

    metrics = {
        'roc_auc': roc_auc_score(
            y_eval,
            y_proba
        ),

        'pr_auc': average_precision_score(
            y_eval,
            y_proba
        ),

        'precision': precision_score(
            y_eval,
            y_pred,
            zero_division=0
        ),

        'recall': recall_score(
            y_eval,
            y_pred,
            zero_division=0
        ),

        'f1': f1_score(
            y_eval,
            y_pred,
            zero_division=0
        )
    }

    return metrics

Jumlah kombinasi hyperparameter: 16


In [19]:
all_results = []

best_models = []

outer_fold_predictions = []

for outer_fold, (
    outer_train_idx,
    outer_test_idx
) in enumerate(
    outer_gkf.split(X, y, groups=groups),
    start=1
):

    print(f"\n{'=' * 60}")
    print(f"OUTER FOLD {outer_fold}")
    print(f"{'=' * 60}")

    # =====================================================
    # 1. OUTER SPLIT
    # =====================================================

    X_outer_train = X.iloc[outer_train_idx]
    y_outer_train = y.iloc[outer_train_idx]

    groups_outer_train = groups.iloc[outer_train_idx]

    X_outer_test = X.iloc[outer_test_idx]
    y_outer_test = y.iloc[outer_test_idx]

    print(
        "Outer train:",
        X_outer_train.shape
    )

    print(
        "Outer test:",
        X_outer_test.shape
    )

    print(
        "Train clients:",
        groups_outer_train.nunique()
    )

    print(
        "Test clients:",
        groups.iloc[outer_test_idx].nunique()
    )

    # =====================================================
    # 2. INNER SPLIT
    # =====================================================

    inner_splits = list(
        inner_gkf.split(
            X_outer_train,
            y_outer_train,
            groups=groups_outer_train
        )
    )

    tuning_results = []

    # =====================================================
    # 3. HYPERPARAMETER TUNING
    # =====================================================

    for param_number, params in enumerate(
        parameter_combinations,
        start=1
    ):

        fold_scores = []

        print(
            f"\nParameter {param_number}/"
            f"{len(parameter_combinations)}"
        )

        # -------------------------------------------------
        # Loop inner folds
        # -------------------------------------------------

        for inner_fold, (
            inner_train_idx,
            inner_val_idx
        ) in enumerate(
            inner_splits,
            start=1
        ):

            # Inner train
            X_train = X_outer_train.iloc[
                inner_train_idx
            ]

            y_train = y_outer_train.iloc[
                inner_train_idx
            ]

            # Inner validation
            X_val = X_outer_train.iloc[
                inner_val_idx
            ]

            y_val = y_outer_train.iloc[
                inner_val_idx
            ]

            # -------------------------------------------------
            # Train XGBoost
            # -------------------------------------------------

            model = build_xgb_model(
                params,
                y_train
            )

            model.fit(
                X_train,
                y_train
            )

            # -------------------------------------------------
            # Validation prediction
            # -------------------------------------------------

            y_val_proba = model.predict_proba(
                X_val
            )[:, 1]

            # -------------------------------------------------
            # Calculate PR-AUC / Average Precision
            # -------------------------------------------------

            pr_auc = average_precision_score(
                y_val,
                y_val_proba
            )

            fold_scores.append(pr_auc)

            print(
                f"  Inner Fold {inner_fold}: "
                f"PR-AUC = {pr_auc:.4f}"
            )

        # -------------------------------------------------
        # Average score across inner folds
        # -------------------------------------------------

        mean_pr_auc = np.mean(fold_scores)

        std_pr_auc = np.std(fold_scores)

        tuning_results.append({
            'params': params,
            'mean_pr_auc': mean_pr_auc,
            'std_pr_auc': std_pr_auc,
            'fold_scores': fold_scores
        })

        print(
            f"  Mean PR-AUC: {mean_pr_auc:.4f}"
        )

    # =====================================================
    # 4. SELECT BEST HYPERPARAMETERS
    # =====================================================

    tuning_df = pd.DataFrame(
        tuning_results
    )

    best_result = tuning_df.loc[
        tuning_df['mean_pr_auc'].idxmax()
    ]

    best_params = best_result['params']

    print("\nBest hyperparameters:")
    print(best_params)

    print(
        "Best inner PR-AUC:",
        round(best_result['mean_pr_auc'], 4)
    )

    # =====================================================
    # 5. RETRAIN BEST MODEL
    # =====================================================

    best_model = build_xgb_model(
        best_params,
        y_outer_train
    )

    best_model.fit(
        X_outer_train,
        y_outer_train
    )

    # =====================================================
    # 6. OUTER TEST EVALUATION
    # =====================================================

    test_metrics = evaluate_model(
        best_model,
        X_outer_test,
        y_outer_test
    )

    print("\nOuter test metrics:")

    for metric, value in test_metrics.items():

        print(
            f"{metric}: {value:.4f}"
        )

    # =====================================================
    # 7. STORE RESULTS
    # =====================================================

    all_results.append({
        'outer_fold': outer_fold,

        'best_params': best_params,

        'inner_pr_auc_mean': best_result['mean_pr_auc'],

        'inner_pr_auc_std': best_result['std_pr_auc'],

        **{
            f'test_{key}': value
            for key, value in test_metrics.items()
        }
    })

    best_models.append(best_model)

    # Simpan prediksi outer test untuk analisis lanjutan
    y_test_proba = best_model.predict_proba(
        X_outer_test
    )[:, 1]

    outer_fold_predictions.append(
        pd.DataFrame({
            'outer_fold': outer_fold,
            'client_hash_id': groups.iloc[outer_test_idx].values,
            'y_true': y_outer_test.values,
            'y_proba': y_test_proba
        }, index=outer_test_idx)
    )


OUTER FOLD 1
Outer train: (18163, 56)
Outer test: (26856, 56)
Train clients: 18
Test clients: 1

Parameter 1/16
  Inner Fold 1: PR-AUC = 0.3845
  Inner Fold 2: PR-AUC = 0.3920
  Inner Fold 3: PR-AUC = 0.8127
  Inner Fold 4: PR-AUC = 0.5543
  Mean PR-AUC: 0.5359

Parameter 2/16
  Inner Fold 1: PR-AUC = 0.3800
  Inner Fold 2: PR-AUC = 0.3930
  Inner Fold 3: PR-AUC = 0.7894
  Inner Fold 4: PR-AUC = 0.5519
  Mean PR-AUC: 0.5286

Parameter 3/16
  Inner Fold 1: PR-AUC = 0.3788
  Inner Fold 2: PR-AUC = 0.3928
  Inner Fold 3: PR-AUC = 0.8108
  Inner Fold 4: PR-AUC = 0.5461
  Mean PR-AUC: 0.5321

Parameter 4/16
  Inner Fold 1: PR-AUC = 0.3813
  Inner Fold 2: PR-AUC = 0.3867
  Inner Fold 3: PR-AUC = 0.7918
  Inner Fold 4: PR-AUC = 0.5463
  Mean PR-AUC: 0.5265

Parameter 5/16
  Inner Fold 1: PR-AUC = 0.4043
  Inner Fold 2: PR-AUC = 0.3765
  Inner Fold 3: PR-AUC = 0.7869
  Inner Fold 4: PR-AUC = 0.5584
  Mean PR-AUC: 0.5315

Parameter 6/16
  Inner Fold 1: PR-AUC = 0.3950
  Inner Fold 2: PR-AUC = 

In [20]:
results_df = pd.DataFrame(
    all_results
)

results_df[[
    'outer_fold',
    'inner_pr_auc_mean',
    'inner_pr_auc_std',
    'test_roc_auc',
    'test_pr_auc',
    'test_precision',
    'test_recall',
    'test_f1'
]]

,outer_fold,inner_pr_auc_mean,inner_pr_auc_std,test_roc_auc,test_pr_auc,test_precision,test_recall,test_f1
0,1,0.538977,0.170499,0.639603,0.557193,0.446486,0.977612,0.613006
1,2,0.614607,0.109879,0.565749,0.372841,0.587500,0.013421,0.026242
2,3,0.561464,0.152886,0.701930,0.508123,0.439646,0.934735,0.598018
3,4,0.445514,0.075300,0.542663,0.772932,0.795715,0.634533,0.706042
4,5,0.580695,0.126519,0.537873,0.515108,0.500000,0.228489,0.313648


In [21]:
test_columns = [
    'test_roc_auc',
    'test_pr_auc',
    'test_precision',
    'test_recall',
    'test_f1'
]

summary = results_df[
    test_columns
].agg([
    'mean',
    'std'
])

print(summary)

      test_roc_auc  test_pr_auc  test_precision  test_recall   test_f1
mean      0.597564     0.545239        0.553870     0.557758  0.451391
std       0.071149     0.144908        0.147564     0.426909  0.279370


In [22]:
best_params_df = pd.DataFrame(
    [
        {
            'outer_fold': result['outer_fold'],
            **result['best_params']
        }
        for result in all_results
    ]
)

best_params_df

,outer_fold,colsample_bytree,gamma,learning_rate,max_depth,min_child_weight,n_estimators,reg_alpha,reg_lambda,subsample
0,1,0.8,0,0.10,3,5,200,0,1,0.8
1,2,0.8,0,0.10,3,1,200,0,1,0.8
2,3,0.8,0,0.05,3,5,200,0,1,0.8
3,4,0.8,0,0.10,5,1,200,0,1,0.8
4,5,0.8,0,0.05,5,1,200,0,1,0.8


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.